# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIR² dataset (second primary colorectal cancer in cancer survivors) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and includes comprehensive clinical and molecular data on colorectal cancer survivors with a second primary cancer. All dataset entities are referenced by their `@id` per the Croissant specification.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

### 1.1 Dataset Citation and Metadata
Below are some additional key metadata fields:

In [ ]:
print(f"Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}\nVersion: {metadata.version}\nKeywords: {getattr(metadata, 'keywords', [])}")
print(f"License: {metadata.license}\nCitation: {metadata.cite_as if hasattr(metadata, 'cite_as') else getattr(metadata, 'citeAs', metadata.identifier)}")

## 2. Data Overview
Explore the available record sets, their `@id`s, and contained fields.

The FAIR² dataset contains **one main record set** representing the tabular clinical-molecular data, as well as field and column definitions for all variables.

**Tip:** All entities (record set, field, column) are referenced by their `@id`.

In [ ]:
# List all record sets and their IDs
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # fall back to attribute name used by croissant (usually recordSet, but implementation may expose record_sets attribute)
    record_sets = getattr(metadata, 'recordSet', [])

print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} - Name: {rs.get('name', '[No Name]')}")
    # List fields for each record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for field in fields:
        print(f"    - {field['@id']} ({field.get('name', '[No Name]')})")

### 2.1 Preview a Record
Let's print a sample record (row) from the primary record set. Replace `<main_record_set_id>` below with the actual `@id` (see output above).

_Note: For this dataset, the canonical record set id is likely_:

- `'https://api.app.sen.science/frontiers/7862866/8719ab13-f086-456e-bc1f-466ccfdf33c7'` (example — ensure from above).

In [ ]:
# Find the primary record set @id
main_record_set_id = None
for rs in record_sets:
    # Use the first record set as the main one if only one exists
    main_record_set_id = rs['@id']
    break
print(f"Using record set: {main_record_set_id}")

# Preview a record
for record in dataset.records(record_set=main_record_set_id):
    print(record)
    break  # Print only the first record

## 3. Data Extraction
Load all records from the identified main record set into a Pandas DataFrame for further analysis. 

In [ ]:
# We'll collect all tabular record sets (typically one for FAIR² table data):
record_set_ids = [main_record_set_id]

# Extract all rows from each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show available columns for the main record set
print(f"Columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Display the first few rows
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the clinical dataset: filter records, normalize a numeric field, and group data by a categorical variable.

### Choosing fields by `@id`
Select a numeric field and a grouping field by their `@id`s. We'll show how to look up their human-readable names if desired.

In [ ]:
# List all columns and choose a numeric field and a categorical group field by their @id
df = dataframes[main_record_set_id]

print("=== Available columns ===")
for col in df.columns:
    print(col)

# Example: Suppose these column @ids exist (replace with actual from above if not exactly matching):
numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/6d383bd2-9ede-4ee6-9c9b-4ed50501df86'   # Example: Age at diagnosis
group_field_id   = 'https://api.app.sen.science/frontiers/7862866/73386c70-1b93-4285-8f11-9b2d02e1f14a'  # Example: Sex

# If these @ids are not found, select columns by index or fallback:
if numeric_field_id not in df.columns:
    numeric_field_id = df.columns[0]  # First numeric as fallback
    print(f"Numeric field used: {numeric_field_id}")
if group_field_id not in df.columns:
    group_field_id = df.columns[1]    # Second as fallback
    print(f"Group field used: {group_field_id}")

# Convert numeric field to numeric dtype, coercing errors
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # Example threshold: mean value
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group data by group_field_id and aggregate (mean of numeric):
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data (mean {numeric_field_id}) by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and group comparisons. We'll use the `@id` for all references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Distribution plot for the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group_field_id
if group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we've loaded the FAIR² clinical-molecular dataset from its Croissant schema using the `mlcroissant` library. We've:
- Explored the schema and referenced all entities by `@id`.
- Loaded the tabular data into a Pandas DataFrame.
- Selected fields by `@id` for processing, filtered rows by a numeric threshold, normalized values, and grouped data by a category.
- Generated visualizations for a numeric variable and across a grouping variable.

This workflow can be extended to deeper analyses or modeling while always referencing data fields by their unique `@id` as specified by the Croissant metadata.